# فصل ۱۲: آشنایی با پایگاه داده‌های برداری (Vector Databases)
## بخش ۳: ریاضیات شباهت، معیارهای فاصله و پیکربندی فضاهای برداری در ChromaDB

در سیستم‌های مبتنی بر مدل‌های زبانی (LLM) و پایگاه داده‌های برداری، کامپیوترها کلمات یا معانی را مستقیماً احساس نمی‌کنند؛ بلکه **فواصل هندسی در فضای چندبعدی** را محاسبه می‌کنند.

$$\text{Distance} \downarrow \iff \text{Similarity} \uparrow$$

در این بخش به بررسی جامع دسته‌بندی وکتور دیتابیس‌ها، فرمول‌های ریاضی ۳ معیار اصلی فاصله (**اقلیدسی $L_2$**، **شباهت کسینوسی** و **ضرب داخلی**)، چرایی برتری کسینوس در پردازش متن (NLP)، و نحوه تنظیم پارامتر `hnsw:space` در ChromaDB و LangChain می‌پردازیم.

### ۱. انواع پایگاه‌های داده برداری (Vector DB Ecosystem)

همان‌طور که در دیتابیس‌های سنتی گزینه‌هایی مانند PostgreSQL، MySQL و SQLite داریم، در دنیای بردارها نیز دو دسته کلی ابزار وجود دارد:

| دسته‌بندی | نمونه‌ها | ویژگی‌ها و نقاط قوت | نقاط ضعف یا محدودیت‌ها |
| :--- | :--- | :--- | :--- |
| **۱. دیتابیس‌های اختصاصی (Native Vector DBs)** | **ChromaDB**, Pinecone, Milvus, Weaviate, Qdrant | از ابتدا برای بردارها طراحی شده‌اند؛ پشتیبانی کامل از عملیات CRUD، فیلترینگ متادیتا و پایداری روی دیسک. | نیاز به منابع برای مقیاس‌های بسیار بزرگ. |
| **۲. کتابخانه‌های جستجوی برداری (Vector Libraries)** | **FAISS** (فیس‌بوک/متا), Annoy, ScaNN | فوق‌العاده سریع و بهینه‌سازی‌شده برای جستجوی صرف در حافظه RAM. | فاقد امکانات دیتابیسی کامل (CRUD ناقص، عدم پایداری دیسکی پیش‌فرض؛ با ریست شدن سرور ایندکس از RAM می‌پرد). |

> **چرا در این دوره از ChromaDB استفاده می‌کنیم؟**
> - متن‌باز، رایگان و بدون نیاز به ثبت‌نام یا API Key ابری.
> - راه‌اندازی فوق‌العاده آسان در پایتون (حالت In-Memory برای توسعه و آزمایش، و حالت Persistent برای ذخیره روی دیسک).
> - منطق کاری و مفاهیم ریاضی آن عیناً با غول‌های تجاری مانند Pinecone و Milvus یکسان است.

### ۲. ریاضیات سه معیار اصلی سنجش شباهت و فاصله

#### ۱. فاصله اقلیدسی (Euclidean Distance / $L_2$)
همان فاصله خط مستقیم هندسی بر پایه قضیه فیثاغورس:

$$d(A, B) = \sqrt{\sum_{i=1}^n (A_i - B_i)^2}$$

- **کاربرد:** تشخیص چهره، بینایی ماشین، داده‌های جغرافیایی (GIS) و مواردی که اندازه بردار اهمیت فیزیکی دارد.
- **مشکل در NLP:** به **طول بردار (Magnitude)** وابسته است. اگر یک جمله دوبار پشت سر هم تکرار شود، معنی یکسان است اما طول بردار بزرگتر شده و در فضای اقلیدسی فاصله زیادی پیدا می‌کند!

---

#### ۲. شباهت کسینوسی (Cosine Similarity)
زاویه بین دو بردار را می‌سنجد و طول بردارها را خنثی (Normalize) می‌کند:

$$\text{Cosine}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|} = \frac{\sum A_i B_i}{\sqrt{\sum A_i^2} \times \sqrt{\sum B_i^2}}$$

- **دامنه مقادیر:** بین `-1.0` (تضاد ۱۸۰ درجه) تا `+1.0` (هم‌جهت و شباهت کامل، زاویه ۰ درجه). عدد `0.0` به معنی زاویه ۹۰ درجه و عدم ارتباط است.
- **فاصله کسینوسی در وکتور دیتابیس‌ها:** $\text{Cosine Distance} = 1 - \text{Cosine Similarity}$
- **چرا در NLP پادشاه است؟** جهت بردار حامل «معنا» است. یک مقاله ۵۰۰ کلمه‌ای و یک توییت ۵ کلمه‌ای در مورد هوش مصنوعی، هر دو به یک جهت اشاره می‌کنند.

#### ۳. ضرب داخلی (Dot Product / Inner Product - IP)
ساده‌ترین و سریع‌ترین عملیات ریاضی حاصل‌ضرب مؤلفه‌ها:

$$A \cdot B = \sum_{i=1}^n A_i \times B_i$$

- **قانون طلایی:** اگر طول بردارها نرمالایز شده باشد ($\|A\| = \|B\| = 1$):
  $$\text{Cosine}(A, B) = \frac{A \cdot B}{1 \times 1} = A \cdot B$$
- بنابراین با **نرمال‌سازی بردارها**، محاسبه ضرب داخلی دقیقاً همان نتیجه شباهت کسینوسی را می‌دهد، ولی بدون هزینه جذر گرفتن یا تقسیم!
- اکثر مدل‌های مدرن (نظیر مدل‌های جدید OpenAI یا مدل‌های ترنسفورمر چندزبانه) بردارهای با طول ۱ تولید می‌کنند تا با ضرب داخلی بیشترین سرعت را به دست آوریم.

### ۳. شبیه‌سازی دستی ریاضیات ۳ معیار با NumPy

در سلول زیر توابع ریاضی این ۳ معیار را پیاده کرده و روی دو بردار نمونه تست می‌کنیم:

In [1]:
import numpy as np

def euclidean_distance(a, b):
    """فاصله اقلیدسی L2"""
    return np.sqrt(np.sum((a - b) ** 2))

def cosine_similarity(a, b):
    """شباهت کسینوسی (بین -1 تا 1)"""
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot / (norm_a * norm_b)

def cosine_distance(a, b):
    """فاصله کسینوسی (1 منهای شباهت کسینوسی)"""
    return 1.0 - cosine_similarity(a, b)

def dot_product(a, b):
    """ضرب داخلی"""
    return np.dot(a, b)

# تست روی دو بردار فرضی ۲ بعدی
vec1 = np.array([3.0, 4.0])        # طول بردار = 5
vec2 = np.array([6.0, 8.0])        # دقیقا در همان جهت اما با طول دو برابر = 10
vec3 = np.array([-4.0, 3.0])       # بردار عمود بر بردار اول (زاویه 90 درجه)

print("--- مقایسه بردار ۱ با بردار ۲ (هم‌جهت با طول‌های متفاوت) ---")
print(f"فاصله اقلیدسی L2:      {euclidean_distance(vec1, vec2):.4f} (فاصله غیرصفر به خاطر طول!)")
print(f"شباهت کسینوسی:        {cosine_similarity(vec1, vec2):.4f} (شباهت ۱۰۰٪ و بی‌نقص)")
print(f"فاصله کسینوسی:        {cosine_distance(vec1, vec2):.4f} (فاصله صفر)")
print(f"ضرب داخلی:            {dot_product(vec1, vec2):.4f}\n")

print("--- مقایسه بردار ۱ با بردار ۳ (عمود بر هم) ---")
print(f"شباهت کسینوسی:        {cosine_similarity(vec1, vec3):.4f} (بی‌ارتباط - زاویه ۹۰ درجه)")
print(f"فاصله کسینوسی:        {cosine_distance(vec1, vec3):.4f}")

--- مقایسه بردار ۱ با بردار ۲ (هم‌جهت با طول‌های متفاوت) ---
فاصله اقلیدسی L2:      5.0000 (فاصله غیرصفر به خاطر طول!)
شباهت کسینوسی:        1.0000 (شباهت ۱۰۰٪ و بی‌نقص)
فاصله کسینوسی:        0.0000 (فاصله صفر)
ضرب داخلی:            50.0000

--- مقایسه بردار ۱ با بردار ۳ (عمود بر هم) ---
شباهت کسینوسی:        0.0000 (بی‌ارتباط - زاویه ۹۰ درجه)
فاصله کسینوسی:        1.0000


### ۴. آزمایش عملی در پردازش متن: چرا $L_2$ در NLP فریب می‌خورد؟

سه متن زیر را در نظر بگیرید:
1. `text_base`: «من سیب دوست دارم.»
2. `text_repeated`: «من سیب دوست دارم. من سیب دوست دارم. من سیب دوست دارم.» (دقیقاً همان معنا اما با کلمات تکراری و طول بیشتر)
3. `text_unrelated`: «قیمت سهام در بازار بورس امروز سقوط کرد.» (موضوع کاملاً بی‌ربط مالی)

حال این متون را بردارسازی کرده و فاصله اقلیدسی و کسینوسی را با هم مقایسه می‌کنیم:

In [2]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

# بارگذاری مدل چندزبانه
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

text_base = "من سیب دوست دارم."
text_repeated = "من سیب دوست دارم. من سیب دوست دارم. من سیب دوست دارم."
text_unrelated = "قیمت سهام در بازار بورس امروز سقوط کرد."

# تبدیل به بردار
v_base = np.array(embedding_model.embed_query(text_base))
v_repeated = np.array(embedding_model.embed_query(text_repeated))
v_unrelated = np.array(embedding_model.embed_query(text_unrelated))

print(f"فاصله اقلیدسی بین متن پایه و متن تکرار شده: {euclidean_distance(v_base, v_repeated):.4f}")
print(f"فاصله اقلیدسی بین متن پایه و متن بی‌ربط:    {euclidean_distance(v_base, v_unrelated):.4f}")
print("--------------------------------------------------")
print(f"فاصله کسینوسی بین متن پایه و متن تکرار شده: {cosine_distance(v_base, v_repeated):.4f} (بسیار نزدیک به صفر!)")
print(f"فاصله کسینوسی بین متن پایه و متن بی‌ربط:    {cosine_distance(v_base, v_unrelated):.4f} (فاصله زیاد و تمایز دقیق)")

d:\Project\LLM\llm-apps-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1872.34it/s]


فاصله اقلیدسی بین متن پایه و متن تکرار شده: 2.4249
فاصله اقلیدسی بین متن پایه و متن بی‌ربط:    7.4784
--------------------------------------------------
فاصله کسینوسی بین متن پایه و متن تکرار شده: 0.0888 (بسیار نزدیک به صفر!)
فاصله کسینوسی بین متن پایه و متن بی‌ربط:    0.9881 (فاصله زیاد و تمایز دقیق)


### ۵. تنظیم معیار فاصله در ChromaDB با پارامتر `hnsw:space`

در ChromaDB می‌توانید هنگام ساخت یک مجموعه (Collection)، متریک فضای فاصله را تعیین کنید:

- `"l2"`: پیش‌فرض سنتی، فاصله مربعی اقلیدسی (Squared $L_2$)
- `"cosine"`: فاصله کسینوسی (ایده‌آل‌ترین انتخاب برای متن و LLM)
- `"ip"`: ضرب داخلی (Inner Product - فوق‌العاده سریع برای بردارهای نرمال‌شده)

ابتدا نحوه استفاده در **کلاینت بومی ChromaDB** را می‌بینیم:

In [3]:
import chromadb

# ایجاد کلاینت موقت در حافظه (Ephemeral Client)
chroma_client = chromadb.Client()

# ۱. ساخت کالکشن با فضای کسینوسی
cosine_collection = chroma_client.create_collection(
    name="my_cosine_collection",
    metadata={"hnsw:space": "cosine"}  # <--- تنظیم فضای هندسی
)

# ۲. ساخت کالکشن با فضای اقلیدسی L2
l2_collection = chroma_client.create_collection(
    name="my_l2_collection",
    metadata={"hnsw:space": "l2"}
)

print("کالکشن کسینوسی:", cosine_collection.metadata)
print("کالکشن اقلیدسی:", l2_collection.metadata)

کالکشن کسینوسی: {'hnsw:space': 'cosine'}
کالکشن اقلیدسی: {'hnsw:space': 'l2'}


### ۶. تنظیم پارامتر `hnsw:space` در LangChain

هنگام استفاده از پکیج `langchain_chroma`، می‌توانید پارامتر `collection_metadata` را پاس بدهید تا فضای شباهت دقیقاً روی کسینوس (`cosine`) یا ضرب داخلی تنظیم شود:

In [4]:
from langchain_chroma import Chroma

docs = [
    "مدل‌های زبانی بزرگ با مکانیزم توجه ترنسفورمر آموزش می‌بینند.",
    "پایگاه‌داده‌های برداری امکان بازیابی معنایی اطلاعات را فراهم می‌سازند.",
    "ورزش صبحگاهی موجب بهبود سلامت قلب و عروق و کاهش استرس می‌شود."
]

# ساخت پایگاه برداری LangChain با تعیین صریح فضای کسینوسی
vector_store_cosine = Chroma(
    collection_name="langchain_cosine_store",
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}  # <--- فاصله کسینوسی
)

# افزودن متن‌ها به پایگاه
vector_store_cosine.add_texts(docs)

# جستجوی شباهت همراه با امتیاز فاصله
query = "چگونه وکتور دیتابیس‌ها جستجوی معنایی انجام می‌دهند؟"
results = vector_store_cosine.similarity_search_with_score(query, k=2)

print(f"🔍 سوال: '{query}'\n")
for idx, (doc, score) in enumerate(results, 1):
    print(f"نتیجه {idx}:")
    print(f"  متن: {doc.page_content}")
    print(f"  فاصله کسینوسی (Cosine Distance): {score:.4f}")
    print(f"  تخمین شباهت کسینوسی (Similarity): {1.0 - score:.4f}\n")

🔍 سوال: 'چگونه وکتور دیتابیس‌ها جستجوی معنایی انجام می‌دهند؟'

نتیجه 1:
  متن: مدل‌های زبانی بزرگ با مکانیزم توجه ترنسفورمر آموزش می‌بینند.
  فاصله کسینوسی (Cosine Distance): 0.5335
  تخمین شباهت کسینوسی (Similarity): 0.4665

نتیجه 2:
  متن: پایگاه‌داده‌های برداری امکان بازیابی معنایی اطلاعات را فراهم می‌سازند.
  فاصله کسینوسی (Cosine Distance): 0.6755
  تخمین شباهت کسینوسی (Similarity): 0.3245



### 🎯 جمع‌بندی نکات کلیدی این درس‌نامه:

1. **پایگاه‌های اختصاصی (Native) در برابر کتابخانه‌ها (Libraries):**
   - دیتابیس‌هایی مثل **ChromaDB** و **Pinecone** پایداری دیسک و عملیات کامل CRUD دارند.
   - کتابخانه‌هایی مثل **FAISS** سرعت بالایی دارند اما در صورت قطعی سیستم، ایندکس مقیم حافظه آن‌ها پاک می‌شود.
2. **معیارهای فاصله:**
   - **اقلیدسی ($L_2$):** حساس به طول و مقیاس بردار (نامناسب برای تغییر طول متون در NLP).
   - **کسینوسی (Cosine):** استاندارد طلایی NLP زیرا زاویه و جهت بردار را که حامل مفهوم است می‌سنجد.
   - **ضرب داخلی (Dot Product / IP):** با نرمالایز کردن بردارها، عیناً نتیجه کسینوسی را با سریع‌ترین سرعت ممکن تولید می‌کند.
3. **تنظیم `hnsw:space`:** در ChromaDB به صورت پیش‌فرض یا صریح می‌توان فضا را با `"cosine"` تنظیم کرد تا نتایج جستجوی متنی دقیق‌ترین معنا را منعکس کنند.